In [4]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

In [5]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['date'] = timestamp_pl.dt.date
df.sample(2)


,timestamp,duration,app,title,status,category,osoba,device,timestamp_pl,date
3588,2025-12-21 20:37:03.744000+00:00,1.027,explorer.exe,,not-afk,studia,AK,PC,2025-12-21 21:37:03.744000+01:00,2025-12-21
5900,2025-12-29 20:01:05.676000+00:00,4.061,chrome.exe,,not-afk,oglądanie filmów,AK,PC,2025-12-29 21:01:05.676000+01:00,2025-12-29


In [23]:
Suma_z_dnia = df.groupby(['osoba','date'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] = round(Suma_z_dnia['duration'] / 3600,2)
Suma_z_dnia = Suma_z_dnia.sort_values(['osoba','date'])
Suma_z_dnia.head()

,osoba,date,duration
0,A,2025-12-13,5.76
1,A,2025-12-14,8.28
2,A,2025-12-15,14.40
3,A,2025-12-16,5.25
4,A,2025-12-17,5.47


In [ ]:
# Wszystkie dni z zakresu
daty = pd.date_range(df['date'].min(), df['date'].max())

In [18]:
# zakładam, że df['date'] jest typu datetime
fig = px.line(
    Suma_z_dnia,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności (godz.)', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna'
)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_layout(xaxis_tickangle=-45)  # dla czytelności etykiet dat

fig.show()
